# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [47]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}


print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [ ]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [49]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [
        {
            "role": "user",
            "content": (
                "Extract the company, role, and years_experience_required from "
                "the job posting below. Return only a valid JSON object with "
                'exactly these keys: "company", "role", and '
                '"years_experience_required". Use null when a field is not '
                "explicitly stated. For years_experience_required, return the "
                "required number of years as a number, or null.\n\n"
                f"Job posting:\n{snippet_text}"
            ),
        }
    ]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    return [
        {
            "role": "user",
            "content": (
                "Extract company, role, and years_experience_required from job "
                "postings. Return only a JSON object with exactly those keys. "
                "Use null when not explicitly stated.\n\n"
                "Example 1:\n"
                "Acme is looking for a Senior Python Developer with at least "
                "5 years of experience.\n"
                'Output: {"company": "Acme", "role": "Senior Python Developer", '
                '"years_experience_required": 5}\n\n'
                "Example 2:\n"
                "The hiring team at Northstar seeks a Product Analyst for its "
                "remote analytics team.\n"
                'Output: {"company": "Northstar", "role": "Product Analyst", '
                '"years_experience_required": null}\n\n'
                "Now extract from this posting:\n"
                f"{snippet_text}\n\n"
            ),
        }
    ]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    return [
        {
            "role": "system",
            "content": (
                "You are an expert recruitment information extractor. Extract "
                "only facts explicitly stated in the job posting. Never invent "
                "values. Return valid JSON with no Markdown and no commentary "
                "using this exact JSON schema:\n"
                "{\n"
                '  "type": "object",\n'
                '  "required": ["company", "role", "years_experience_required"],\n'
                '  "additionalProperties": false,\n'
                '  "properties": {\n'
                '    "company": {"type": ["string", "null"]},\n'
                '    "role": {"type": ["string", "null"]},\n'
                '    "years_experience_required": {"type": ["number", "null"]}\n'
                "  }\n"
                "}\n"
                "Use null for information that is not stated."
            ),
        },
        {
            "role": "user",
            "content": f"Job posting:\n{snippet_text}",
        },
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            "role": "user",
            "content": (
                "Analyze the job posting step by step: identify the explicit "
                "company name, the explicit role or job title, and the minimum "
                "years of experience required. Do not infer or fabricate values.\n\n"
                "After your analysis, return only a valid JSON object with "
                'exactly these keys: "company", "role", and '
                '"years_experience_required". Use null when not stated. '
                "years_experience_required must be a number or null. Do not "
                "include your analysis in the JSON or final response.\n\n"
                f"Job posting:\n{snippet_text}"
            ),
        }
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [57]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    text =  text.strip()
    
     # Remove an optional Markdown code fence, such as ```json ... ```.
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    # First try parsing the entire response.
    try:
        result = json.loads(text)
        return result if isinstance(result, dict) else None
    except json.JSONDecodeError:
        pass

    # If there's extra text around the JSON, try parsing an object from each
    # opening brace until one succeeds.
    decoder = json.JSONDecoder()
    for index, character in enumerate(text):
        if character == "{":
            try:
                result, _ = decoder.raw_decode(text[index:])
                if isinstance(result, dict):
                    return result
            except json.JSONDecodeError:
                continue

    return None

def compute_cost_usd(model: str, prompt_tokens: int, completion_tokens: int) -> float:
    """Convert a usage tuple into USD.

    Returns 0.0 for unknown models rather than raising — a missing rate
    shouldn't break a batch run; it just means we can't price that call.
    """
    rates = RATES[model]
    if rates is None:
        return 0.0
    
    return (prompt_tokens * rates['in'] + completion_tokens * rates['out']) 

async def run_one(strategy_name: str, snippet: dict) -> dict:
    
    snippet_text = snippet
    
    message = STRATEGIES[strategy_name](snippet_text)

    #print (f" Message is ::: {message}")
    
    start_time = time.perf_counter()
    response = await client.chat.completions.create(
    model=MODEL,
    messages=message,
    temperature=TEMPERATURE)

    #Get the timedifference
    latency_s = time.perf_counter() - start_time

    #Usage calculation
    prompt_tokens = response.usage.prompt_tokens if response.usage else 0
    completion_tokens = response.usage.completion_tokens if response.usage else 0
    rates = RATES[MODEL]
    #cost_usd = prompt_tokens * rates["in"] + completion_tokens * rates["out"]
    cost_usd = compute_cost_usd(MODEL, prompt_tokens, completion_tokens)
    
    
    # Parse the response received from OpenAI
    raw_response = response.choices[0].message.content
    parsed_extraction = parse_response(raw_response)

    return {
        "strategy": strategy_name,
        "snippet_id": snippet["id"],
        "raw_response": raw_response,
        "extracted": parsed_extraction,
        "cost_usd": round(cost_usd, 6),
        "latency_s": latency_s,
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    results = await asyncio.gather(*tasks)
    #print(f"Completed {len(results)} calls.")
    return results

In [58]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')


Got 40 results.


## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [59]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if extracted is None:
        return 0
    fields = ("company", "role", "years_experience_required")
    matches = 0

    for field in fields:
        extracted_value = extracted.get(field)
        gold_value = gold.get(field)

        if isinstance(extracted_value, str) and isinstance(gold_value, str):
            extracted_value = extracted_value.strip().lower()
            gold_value = gold_value.strip().lower()

        if extracted_value == gold_value:
            matches += 1

    return matches


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    fields = ("company", "role", "years_experience_required")
    gold_values = {field: gold.get(field) for field in fields}
    extracted_values = {field: extracted.get(field) for field in fields}

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an impartial evaluator of job-posting extraction. "
                    "Compare the extracted values with the gold values, using "
                    "the job posting as context. Treat capitalization and "
                    "surrounding whitespace as insignificant for strings. "
                    "Do not give credit for unsupported or fabricated values. "
                    "Score using this rubric: 4 = all three fields correct; "
                    "3 = two fields correct and no fabricated data; "
                    "2 = one field correct, or a field is fabricated; "
                    "1 = none correct or the extraction is unusable. "
                    'Return only a JSON object in this form: {"score": 1}.'
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Job posting:\n{snippet_text}\n\n"
                    f"Gold values:\n{json.dumps(gold_values, ensure_ascii=False)}\n\n"
                    f"Extracted values:\n"
                    f"{json.dumps(extracted_values, ensure_ascii=False)}"
                ),
            },
        ],
    )

    judge_text = response.choices[0].message.content
    judge_result = parse_response(judge_text)
    #print ("Judge result::", judge_result)
    try:
        score = int(judge_result["score"]) if judge_result else 1
    except (KeyError, TypeError, ValueError):
        return 1

    return score if score in (1, 2, 3, 4) else 1


In [63]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
scored = []   # list of result dicts with scoring fields added

snippets_by_id = {snippet["id"]: snippet for snippet in snippets}

async def score_result(item: dict) -> dict:
    snippet_id = item["snippet_id"]
    extracted = item["extracted"]
    gold = golden[snippet_id]

    scored_item = dict(item)
    scored_item["accuracy"] = score_accuracy(extracted, gold)
    scored_item["parse_success"] = extracted is not None
    scored_item["llm_judge_score"] = await score_llm_judge(
        snippets_by_id[snippet_id]["snippet"],
        extracted,
        gold,
    )
    return scored_item

scored = await asyncio.gather(*(score_result(item) for item in results))
print(f'Scored {len(scored)} results.')


#for item in results:


Scored 40 results.


## Step 5 — Build the comparison table

In [64]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(7)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
"""
summary[["accuracy", "parse_success", "llm_judge_score", "latency_s"]] = (
    summary[["accuracy", "parse_success", "llm_judge_score", "latency_s"]].round(3)
)
summary["cost_usd"] = summary["cost_usd"].round(6)
"""
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.7,1.0,3.9,0.000415,1.227816
few_shot,2.7,1.0,3.9,0.000436,1.286313
structured,2.8,1.0,4.0,0.000476,1.236716
zero_shot,2.7,1.0,3.9,0.000404,1.412237


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```